# Klasifikasi Bagian Tanaman — MobileNetV2 Transfer Learning, 5 Kelas

**Dataset:** `dataset_tanaman_final_5kelas.zip` (52.500 gambar, masing-masing 10.500 untuk `akar`, `batang`, `buah`, `bunga`, `daun`). **Pembagian:** train 7.350, validation 1.575, test 1.575 per kelas. **Input:** RGB 224×224 (`0–255`). **Arsitektur:** `MobileNetV2` (ImageNet *pretrained* + *fine-tuning* dengan *label smoothing* & augmentasi kontras/kecerahan). **Label:** urutan alfabet otomatis dari direktori.

## Sebelum menjalankan
1. Upload file `dataset_tanaman_final_5kelas.zip` ke **MyDrive** di Google Drive (atau ke `/content/` di panel Files Colab).
2. Di Google Colab pilih **Runtime → Change runtime type → T4 GPU**.
3. Jalankan semua cell dari atas ke bawah.

## 1. Hubungkan Google Drive dan import library

In [ ]:
from google.colab import drive, files
from pathlib import Path
import csv
import json
import zipfile
import shutil

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EXPECTED_CLASSES = ["akar", "batang", "buah", "bunga", "daun"]
EXPECTED_PER_CLASS = {"train": 7350, "val": 1575, "test": 1575}

tf.keras.utils.set_random_seed(SEED)
if not Path('/content/dataset_tanaman_final_5kelas.zip').is_file():
    drive.mount('/content/drive')
else:
    print('Menggunakan /content/dataset_tanaman_final_5kelas.zip tanpa mount Drive.')
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Ekstrak ZIP dataset langsung ke Colab
Simpan file ZIP pada `MyDrive/dataset_tanaman_final_5kelas.zip`. Dataset diekstrak ke disk Colab agar pembacaan gambar lebih cepat dibanding membaca file satu per satu dari Google Drive.

In [ ]:
ZIP_LOCAL = Path('/content/dataset_tanaman_final_5kelas.zip')
ZIP_DRIVE = Path('/content/drive/MyDrive/dataset_tanaman_final_5kelas.zip')
FOLDER_DRIVE = Path('/content/drive/MyDrive/dataset_tanaman_final')
BASE_LOCAL = Path('/content/dataset_tanaman_final')

zip_source = None
if ZIP_LOCAL.is_file() and ZIP_LOCAL.stat().st_size > 0:
    zip_source = ZIP_LOCAL
    print(f'Menggunakan ZIP lokal Colab: {zip_source} ({zip_source.stat().st_size:,} bytes)')
elif ZIP_DRIVE.is_file():
    drive_size = ZIP_DRIVE.stat().st_size
    print(f'Menyalin ZIP dari Google Drive ({drive_size:,} bytes) ke disk Colab...')
    shutil.copy2(ZIP_DRIVE, ZIP_LOCAL)
    zip_source = ZIP_LOCAL

if zip_source is not None:
    size_bytes = zip_source.stat().st_size
    with open(zip_source, 'rb') as f:
        magic = f.read(4)
    if not zipfile.is_zipfile(zip_source):
        raise zipfile.BadZipFile(
            f'{zip_source} bukan ZIP valid (ukuran={size_bytes:,} bytes, header={magic!r}). '
            'Jika baru selesai upload ke Drive, jalankan Runtime -> Restart session atau tunggu sinkronisasi selesai (~682.2 MB / 715,365,960 bytes).'
        )
    if BASE_LOCAL.exists():
        shutil.rmtree(BASE_LOCAL)
    with zipfile.ZipFile(zip_source) as z:
        z.extractall('/content')
    if (BASE_LOCAL / 'train').is_dir():
        BASE = BASE_LOCAL
    elif (BASE_LOCAL / 'dataset_tanaman_final' / 'train').is_dir():
        BASE = BASE_LOCAL / 'dataset_tanaman_final'
    elif Path('/content/train').is_dir():
        BASE = Path('/content')
    else:
        BASE = BASE_LOCAL
    print('Dataset diekstrak ke:', BASE)
elif FOLDER_DRIVE.is_dir():
    BASE = FOLDER_DRIVE
    print('Menggunakan folder dari Google Drive:', BASE)
else:
    raise FileNotFoundError(
        'Upload dataset_tanaman_final_5kelas.zip ke MyDrive atau ke folder /content Colab'
    )

for split, expected in EXPECTED_PER_CLASS.items():
    print('\nSplit:', split)
    for cls in EXPECTED_CLASSES:
        folder = BASE / split / cls
        paths = list(folder.glob('*.jpg'))
        print(f'  {cls:8s}: {len(paths):5d}')
        assert len(paths) == expected, f'{split}/{cls}: diharapkan {expected}, ditemukan {len(paths)}'

print('\nDataset lengkap: 5 kelas, 52.500 gambar.')

## 3. Load dataset sebagai TensorFlow Dataset
Label kelas dibaca otomatis dari urutan folder alfabet (`akar`, `batang`, `buah`, `bunga`, `daun`). **Preprocessing skala `[-1, 1]` untuk MobileNetV2 ditempatkan di dalam model** (`Rescaling(1./127.5, offset=-1.0)`), bukan di loader maupun di FastAPI, sehingga FastAPI tetap mengirim piksel RGB `0–255`.

In [ ]:
def get_ds(split: str, shuffle: bool):
    return tf.keras.utils.image_dataset_from_directory(
        BASE / split,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED if shuffle else None,
        label_mode='categorical',
        color_mode='rgb',
        interpolation='bilinear',
    )

train_ds_raw = get_ds('train', True)
val_ds_raw = get_ds('val', False)
test_ds_raw = get_ds('test', False)

class_names = list(train_ds_raw.class_names)
assert class_names == EXPECTED_CLASSES, f'Urutan kelas tidak cocok: {class_names}'
assert list(val_ds_raw.class_names) == class_names
assert list(test_ds_raw.class_names) == class_names

AUTO = tf.data.AUTOTUNE
train_ds = train_ds_raw.prefetch(AUTO)
val_ds = val_ds_raw.prefetch(AUTO)
test_ds = test_ds_raw.prefetch(AUTO)

print('Urutan kelas:', class_names)
print('Jumlah kelas:', len(class_names))

## 4. Lihat contoh data sebelum training

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
images, labels = next(iter(train_ds_raw))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].numpy().astype('uint8'))
    ax.set_title(class_names[int(np.argmax(labels[i]))])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Model MobileNetV2 (Transfer Learning + Anti-Bias Augmentasi)
Lapisan augmentasi (`RandomFlip`, `RandomRotation`, `RandomZoom`, `RandomContrast`, `RandomBrightness`) hanya aktif saat training agar model lebih tahan terhadap variasi pencahayaan dan *framing* dunia nyata. Lapisan `Rescaling(1./127.5, offset=-1.0)` ada **di dalam model** sehingga FastAPI tetap mengirim piksel RGB `0–255`.

In [ ]:
from tensorflow.keras import layers

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,), name='input_image')
x = layers.RandomFlip('horizontal')(inputs)
x = layers.RandomRotation(0.08)(x)
x = layers.RandomZoom(0.15)(x)
x = layers.RandomContrast(0.15)(x)
x = layers.RandomBrightness(0.15)(x)
x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name='mobilenetv2_rescale')(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)

model = tf.keras.Model(inputs, outputs, name='mobilenetv2_bagian_tanaman_5_kelas')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy'],
)
model.summary()

## 6. Training 2 Tahap (Feature Extraction + Fine-Tuning)
- **Tahap 1 (Feature Extraction):** melatih *head* klasifikasi dengan backbone beku.
- **Tahap 2 (Fine-Tuning):** membuka lapisan atas `MobileNetV2` (mulai layer ke-100) dengan `learning_rate` kecil (`5e-5`) sementara lapisan `BatchNormalization` tetap dibekukan agar statistik ImageNet stabil.

Data **test** tidak pernah dipakai untuk memilih model atau hyperparameter.

In [ ]:
INITIAL_EPOCHS = 6
FINE_TUNE_EPOCHS = 10
FINE_TUNE_AT = 110

callbacks_stage1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=3, restore_best_weights=True
    ),
]

print('=== Tahap 1: Feature Extraction (backbone dibekukan) ===')
history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    callbacks=callbacks_stage1,
)

# Tahap 2: Fine-tuning lapisan atas MobileNetV2 (BatchNorm tetap dibekukan)
base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy'],
)

callbacks_stage2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=3, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6
    ),
]

print('=== Tahap 2: Fine-Tuning MobileNetV2 ===')
history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=len(history_stage1.history['loss']) + FINE_TUNE_EPOCHS,
    initial_epoch=len(history_stage1.history['loss']),
    callbacks=callbacks_stage2,
)

## 7. Grafik hasil training

In [ ]:
RESULTS = Path('/content/results_mobilenetv2_5kelas')
RESULTS.mkdir(parents=True, exist_ok=True)

acc = history_stage1.history['accuracy'] + history_stage2.history['accuracy']
val_acc = history_stage1.history['val_accuracy'] + history_stage2.history['val_accuracy']
loss = history_stage1.history['loss'] + history_stage2.history['loss']
val_loss = history_stage1.history['val_loss'] + history_stage2.history['val_loss']
ft_epoch = len(history_stage1.history['accuracy']) - 1

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(acc, label='Train')
ax[0].plot(val_acc, label='Validation')
ax[0].axvline(ft_epoch, color='gray', linestyle='--', label='Mulai Fine-Tuning')
ax[0].set_title('Accuracy (MobileNetV2)')
ax[0].set_xlabel('Epoch')
ax[0].legend()

ax[1].plot(loss, label='Train')
ax[1].plot(val_loss, label='Validation')
ax[1].axvline(ft_epoch, color='gray', linestyle='--', label='Mulai Fine-Tuning')
ax[1].set_title('Loss (MobileNetV2)')
ax[1].set_xlabel('Epoch')
ax[1].legend()
plt.tight_layout()
plt.savefig(RESULTS / 'training_curves.png', dpi=160, bbox_inches='tight')
plt.show()

## 8. Evaluasi satu kali pada test set
Hasil evaluasi test dapat dipakai pada paper UAS: accuracy, precision, recall, F1-score, dan confusion matrix.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
y_true = np.argmax(np.concatenate([y.numpy() for _, y in test_ds], axis=0), axis=1)
y_prob = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

report_text = classification_report(
    y_true, y_pred, target_names=class_names, digits=4, zero_division=0
)
report_dict = classification_report(
    y_true, y_pred, target_names=class_names,
    digits=4, zero_division=0, output_dict=True
)
print('Test accuracy:', round(float(test_accuracy), 4))
print(report_text)
(RESULTS / 'classification_report.txt').write_text(report_text, encoding='utf-8')
(RESULTS / 'metrics.json').write_text(
    json.dumps({'test_loss': float(test_loss), 'test_accuracy': float(test_accuracy),
                'classification_report': report_dict}, indent=2, ensure_ascii=False),
    encoding='utf-8'
)

cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(cmap='Blues', ax=ax, colorbar=False)
plt.title('Confusion Matrix - MobileNetV2 (5 Kelas)')
plt.tight_layout()
plt.savefig(RESULTS / 'confusion_matrix.png', dpi=160, bbox_inches='tight')
plt.show()

## 9. Simpan model untuk FastAPI
Keluarkan **dua file utama**: `model_tanaman.keras` dan `class_names.json`. Backend dari starter project sebelumnya sudah membaca urutan kelas secara dinamis; jumlah output model harus 5.

In [ ]:
EXPORT = Path('/content/export_model_tanaman_5kelas')
EXPORT.mkdir(parents=True, exist_ok=True)
model.save(EXPORT / 'model_tanaman.keras')
(EXPORT / 'class_names.json').write_text(
    json.dumps(class_names, ensure_ascii=False, indent=2), encoding='utf-8'
)

# Uji file yang diekspor, sebelum digunakan oleh FastAPI.
loaded = tf.keras.models.load_model(EXPORT / 'model_tanaman.keras', compile=False)
assert int(loaded.output_shape[-1]) == len(class_names) == 5
print('Model diekspor, urutan label:', class_names)

export_zip = Path('/content/model_tanaman_export_5kelas.zip')
with zipfile.ZipFile(export_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(EXPORT / 'model_tanaman.keras', arcname='model_tanaman.keras')
    z.write(EXPORT / 'class_names.json', arcname='class_names.json')
    for path in RESULTS.iterdir():
        if path.is_file():
            z.write(path, arcname=f'results/{path.name}')
print('File untuk diunduh:', export_zip)
files.download(str(export_zip))

## 10. Integrasi dengan FastAPI + Next.js

Ekstrak `model_tanaman_export_5kelas.zip`, kemudian salin `model_tanaman.keras` dan `class_names.json` ke folder **`backend/models/`** dari proyek Anda.

```text
backend/models/
  model_tanaman.keras
  class_names.json
```

Jalankan backend dari folder `backend`:

```powershell
.\.venv\Scripts\python.exe -m uvicorn app.main:app --reload --port 8000
```

Buka `http://127.0.0.1:8000/health` untuk memeriksa model, `http://127.0.0.1:8000/docs` untuk mencoba `POST /predict`, lalu jalankan frontend (`npm run dev`) dan buka `http://localhost:3000`.

**Penting:** jika JSON health masih `model_missing`, artinya kedua file model belum berada pada folder yang benar. Model klasifikasi hanya mengembalikan satu bagian tanaman utama, bukan menandai beberapa organ sekaligus.